# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

### Update System Path

In [2]:
import sys
sys.path.append('../05_src/')

### Use a Logger

In [3]:
from utils.logger import get_logger
_logs = get_logger(__name__, log_dir='../06_logs/')

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

Selected Document

+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)

In [4]:
from langchain_community.document_loaders import PyPDFLoader

try:
    file_path = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    _logs.info('Document loaded successfully. Number of pages: %d', len(docs))
except Exception as e:
    _logs.error('An error occurred while loading the document: %s', str(e))

2026-04-20 18:41:38,591, 1705955870.py, 7, INFO, Document loaded successfully. Number of pages: 26


Join all document pages

In [5]:
document_text = ""

try:
    for page in docs:
        document_text += page.page_content + "\n"
    _logs.info('Document text extracted successfully. Total length: %d characters', len(document_text))
except Exception as e:
    _logs.error('An error occurred while extracting document text: %s', str(e))

2026-04-20 18:41:38,598, 2773480818.py, 6, INFO, Document text extracted successfully. Total length: 53851 characters


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


### Initialize OpenAI API

In [6]:
import os
from openai import OpenAI

try:
    api_key = os.getenv('API_GATEWAY_KEY')
    if not api_key:
        raise ValueError("API_GATEWAY_KEY environment variable is not set.")
    client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                    api_key='any value',
                    default_headers={"x-api-key": api_key})
    _logs.info('OpenAI client initialized successfully.')
except Exception as e:
    _logs.error('An error occurred while initializing OpenAI client: %s', str(e))

2026-04-20 18:41:39,025, 3006793336.py, 11, INFO, OpenAI client initialized successfully.


### Add the Developer Prompt (System Prompt)
Use the tone defined in the system_prompt to answer the prompt.

In [7]:
# The following system prompt was not used in the Summary: <summary> response, because it was conflicing with the instructions in the user prompt. 
# system_prompt = "For every paragraph, use parentheses to include a comedic internal thought or a sarcastic side-note."

# This prompt prevents the model from "cleaning them up" when a concise summary is requested. 
# I also included that number of expected parapraphs (three) in the user prompt.
# system_prompt = f"""Every paragraph MUST contain exactly one set of parentheses containing a sarcastic internal monologue. 
# This is a non-negotiable structural element."""

# I tested this prompt, but I was not getting a good ratio of sports jargon to overall content, so I switched to the one below.
# system_prompt = "Use a tone that is a sports commentator, high energy, informative and with sports jargon. Highlight the sports jargon in bold."

# This prompt was sports jargon but it's difficlt to identify the sports jargon, so I switched back to the one below.
# system_prompt = f"""Use a tone that is a sports commentator, high energy, informative and with sports jargons.
# Every paragraph MUST contain at least two sports jargons. Highlight them in bold.
# This is a non-negotiable structural element."""

# I chose to switch to a sarcastic tone, because it makes the summary more entertaining to read.
system_prompt = f"""Every paragraph MUST contain exactly one set of parentheses containing a sarcastic internal monologue tone. 
This is a non-negotiable structural element."""

### Add the User Prompt

In [8]:
prompt = f"""
    Given the following context from a document, do the following:
    
    1. Identify the document's author and title.
    2. Explain why this article is relevant for an AI professional in their professional development, write no longer than one paragraph.
    3. Summarize concisely and succinctly with no longer than 1000 tokens. 
       Make sure to include the most important details and insights in at least three paragraphs.
        
    The document is the following: 
    <document>
    {document_text}
    </document>

    Provide your response in the following format:
    Author: <author>
    Title: <title>
    Relevance: <relevance>
    Summary: <summary>
"""

### Use the gpt-4o-mini to create the prompt

In [9]:
try:
    response = client.responses.create(
        model = 'gpt-4o-mini', # depending on the tier we have available, we might need to update the model to be used
        instructions = system_prompt,
        input = prompt,
    )
    _logs.info('Response generated successfully.')
except Exception as e:
    _logs.error(f"An error occurred in the response generation: {e}")

2026-04-20 18:42:06,240, 607725725.py, 7, INFO, Response generated successfully.


In [10]:
from IPython.display import display, Markdown

display(Markdown(response.output_text))

**Author:** Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari  
**Title:** The GenAI Divide: State of AI in Business 2025  

**Relevance:** This article is crucial for AI professionals aiming to understand the real challenges of implementing Generative AI in business settings, particularly the stark contrast between high adoption rates and low transformation outcomes. (Who doesn’t want to learn about all the ways AI projects can flop spectacularly?) 

**Summary:**  
The report reveals a significant phenomenon termed the GenAI Divide, where a staggering 95% of organizations are failing to achieve measurable returns on their $30–40 billion investment in Generative AI initiatives. Although a large majority have adopted tools like ChatGPT, they primarily enhance individual productivity rather than drive bottom-line impact. The key barriers to success are not purely technical; rather, they reside in the organizations’ learning capabilities. Systems that lack memory, context-awareness, and adaptation are leading to stalled implementations. Hence, while enterprises are excited about AI, they often become entangled in complex workflows that render their custom solutions ineffective. (Diving headfirst into the abyss, are we?)

Further analysis has identified four main patterns contributing to the GenAI Divide: limited disruption across most industries, the paradox of enterprises leading in pilot counts but lagging in project scale-up, biases in budget allocations favoring visible functions over back-office systems, and an implementation advantage for organizations that partner externally rather than building internally. Notably, success in crossing the divide often stems from the degree of customization and contextual integration of AI tools. In stark contrast, many enterprises are overly focused on generic solutions, failing to tailor systems to their specific workflows. (Just what we need—a cookie-cutter approach that doesn’t fit anyone.)

Curiously, a thriving "shadow AI economy" has emerged, where employees use personal AI tools to enhance their productivity without formal buy-in from organizations. This underlines a vital disconnect between what employees need for efficiency versus what is officially offered. Moreover, investment patterns reflect a troubling bias toward sales and marketing initiatives over potentially more impactful back-office functionalities. Organizations succeeding in their AI endeavors prioritize deep customization, learning capabilities, and strong vendor partnerships. Ultimately, there is an urgent need for these organizations to move beyond static tools requiring constant prompting, thus enabling them to harness the real potential of Generative AI. (Oh yes, let's just do what’s easiest—what could possibly go wrong?)

In [11]:
# response.model_dump()

### Output should be a Pydantic BaseModel object. This instruction is pending until we have more classes.

In [12]:
print(f"""Tone: {response.instructions}""")
print(f"""InputTokens: {response.usage.input_tokens}""")
print(f"""OutputTokens: {response.usage.output_tokens}""")

Tone: Every paragraph MUST contain exactly one set of parentheses containing a sarcastic internal monologue tone. 
This is a non-negotiable structural element.
InputTokens: 10876
OutputTokens: 500


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
